# Revisão humana, aprovação e rollback

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="07_registry_governanca"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.a9291c01',
 'model_version': '1',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'notebooks-a9291c01',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Modelos/versões e pedidos de revisão. Registro é candidato; aprovação exige token separado de operador e versão explícita.

In [2]:
import mlflow
mlflow.set_tracking_uri(cfg.tracking);client=mlflow.MlflowClient()
from urllib.parse import urlencode
catalog=call(cfg,"GET","/v1/governance/models?"+urlencode(dict(prefix=params.registry_prefix,page_token=params.registry_page_token,limit=100)))
display(pd.DataFrame(catalog["items"]));display({"next_page_token":catalog["next_page_token"]})
versions=client.search_model_versions("name='"+params.model_name+"'",max_results=100)
display(pd.DataFrame([dict(model=v.name,version=v.version,run_id=v.run_id,tags=v.tags) for v in versions]))
display(pd.DataFrame(call(cfg,"GET","/v1/governance/requests")))


,model_name,aliases,tags
0,hi.demo.integration.70491a51,{},{}
1,hi.demo.notebook.a9291c01,{},{}
2,hi.demo.probe.18cb82de,{},{}
3,hi.demo.v3-negative.699bb4f2,{},{}
4,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,{},{}
5,hi.pt.demanda.DEMO-C1.DEMO-CR1.P2,{},{}
6,hi.pt.demanda.DEMO-C1.DEMO-CR2.P1,{},{}
7,hi.pt.demanda.DEMO-C1.DEMO-CR2.P2,{},{}
8,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,{},{}
9,hi.pt.demanda.DEMO-C1.DEMO-CR3.P2,{},{}


{'next_page_token': ''}

,model,version,run_id,tags
0,hi.demo.notebook.a9291c01,1,420b1e103c8741ddbb8ec66427a95c0b,"{'solution_id': 'horarios-inteligentes', 'envi..."


,approval_id,model_name,model_version,action,reason,requested_by,reviewed_by,correlation_id,status,prior_version,artifact_sha256,run_id,created_at_utc,reviewed_at_utc
0,1564146d-05f5-41ef-b418-4e51779a86a3,hi.demo.v3-negative.699bb4f2,1,REVOKE,Explicit isolated V3 acceptance,isolated-v3,poc-reviewer,a903f8cf-a102-4efa-8307-12081d330141,REVOKED,1,None,None,2026-11-04T21:42:52.748018+00:00,2026-11-04T21:42:52.769740+00:00
1,d199777e-02ad-43b7-91a6-0feb9cb1eafa,hi.demo.v3-negative.699bb4f2,1,APPROVE,Explicit isolated V3 acceptance,isolated-v3,poc-reviewer,1294aa75-13b6-49d3-853b-b8e15f437ad7,REVOKED,None,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,8de6f68bbf5c41b3a67a461e3d67775a,2026-11-04T21:42:50.410343+00:00,2026-11-04T21:42:50.431901+00:00


2. Inspecionar artefatos e proveniência da versão selecionada. Esperado: hashes iguais; nenhuma aprovação implícita.

In [3]:
if params.model_version:
    from src.operations.training import trace_for
    from src.workbench.verification import verify_model_storage
    trace=trace_for(client,client.get_model_version(params.model_name,params.model_version))
    evidence["candidate"]=trace;evidence["storage"]=verify_model_storage(trace)
    display(trace);display(pd.DataFrame(evidence["storage"]))


{'run_id': '420b1e103c8741ddbb8ec66427a95c0b',
 'experiment_id': '3',
 'model_name': 'hi.demo.notebook.a9291c01',
 'model_version': '1',
 'model_uri': 'models:/hi.demo.notebook.a9291c01/1',
 'artifact_uri': 'models:/m-053c00350db44eb1a28c5dae0ed54a66',
 'artifact_sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
 'metrics': {'mae': 3.758630658273197,
  'rmse': 4.9142272796917865,
  'baseline_mae': 7.466666666666665,
  'baseline_rmse': 8.821497143283464,
  'duration_seconds': 0.5707518139970489,
  'cpu_seconds': 0.22584980599999938,
  'peak_rss_kib': 276956.0},
 'training_execution_id': 'e7cdf170-3841-4794-a1b7-f454f9c7cf2a'}

,bucket,key,bytes,sha256,object_version_id,model_version,status
0,pucpr-poc,mlflow/3/models/m-053c00350db44eb1a28c5dae0ed5...,1336,d1ba43508ec6fc0a1d34d487a39af29294c7af2212affc...,203e4bec-7aef-40ba-92c5-bf1182776cf7,1,PASS
1,pucpr-poc,mlflow/3/models/m-053c00350db44eb1a28c5dae0ed5...,8671,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,8ef84859-a09a-4855-8bfc-85178ba7212f,1,PASS
2,pucpr-poc,mlflow/3/models/m-053c00350db44eb1a28c5dae0ed5...,80,3f22b46a5a2293ca36743e1757d14e9968aa14f9d57a28...,a4aea68c-84c8-4a9a-bf83-206683aadabb,1,PASS


3. Ação opt-in: solicitar revisão ou decidir pedido identificado. APPROVE/REVOKE/ROLLBACK requerem justificativa; reviewer vem da credencial, não do texto digitado.

In [4]:
if params.governance_action!="read":
    params.authorize()
    if params.governance_action=="request":
        if not params.model_version or len(params.reason)<8: raise ValueError("Versão e justificativa obrigatórias")
        evidence["review"]=call(cfg,"POST","/v1/governance/requests",dict(model_name=params.model_name,model_version=params.model_version,
            action=params.review_action,reason=params.reason,requested_by=params.requester))
    else:
        if not params.approval_id: raise ValueError("approval_id obrigatório")
        evidence["review"]=call(cfg,"POST",f"/v1/governance/requests/{params.approval_id}/review",
            dict(approve=params.governance_action=="approve",confirm="REVIEW_EXPLICIT_VERSION"),operator=True)
    display(evidence["review"])
else:
    evidence["review"]="NOT_EXECUTED: leitura"


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [5]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/07_registry_governanca.json'